Nomor 4

In [5]:
import numpy as np
from sklearn.cluster import KMeans
import pandas as pd

data = {
    'X': [1, 4, 7, 10, 13, 10, 15, 30, 45, 60, 100, 110, 125, 100, 150, 30, 30, 40, 50, 25, 200, 220, 220, 200, 150],
    'Y': [2, 5, 8, 11, 14, 20, 20, 35, 50, 65, 110, 120, 135, 100, 160, 20, 25, 25, 60, 30, 150, 150, 150, 260, 170],
    'Z': [3, 6, 9, 12, 15, 30, 25, 40, 55, 70, 120, 130, 140, 100, 170, 10, 25, 40, 30, 40, 210, 200, 210, 240, 200]
}
df = pd.DataFrame(data)
kmean = KMeans(n_clusters=5, random_state=0)
kmean.fit(df)
df['Cluster'] = kmean.labels_ + 1
df_sorted = df.sort_values('Cluster').reset_index(drop=True)
print(df_sorted)


      X    Y    Z  Cluster
0   125  135  140        1
1   100  100  100        1
2   110  120  130        1
3   100  110  120        1
4    60   65   70        1
5    25   30   40        2
6    50   60   30        2
7    40   25   40        2
8    30   25   25        2
9    30   20   10        2
10    1    2    3        2
11   30   35   40        2
12   15   20   25        2
13   10   20   30        2
14   13   14   15        2
15   10   11   12        2
16    7    8    9        2
17    4    5    6        2
18   45   50   55        2
19  200  150  210        3
20  220  150  200        3
21  220  150  210        3
22  150  160  170        4
23  150  170  200        4
24  200  260  240        5


Nomor 5

In [6]:
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping

data = {
    'KD': [90, 85, 90, 80, 75, 90, 50, 100, 85, 50, 75, 80, 50, 90, 90, 75, 90, 50, 80, 60, 95, 80, 50, 80, 75, 80, 50, 90, 60, 95, 80, 85, 85, 90],
    'BI': [80, 90, 75, 80, 50, 80, 60, 95, 80, 55, 85, 45, 75, 80, 50, 90, 90, 75, 80, 50, 90, 60, 95, 80, 50, 85, 90, 90, 85, 80, 70, 80, 60, 60],
    'K': [90, 95, 80, 85, 90, 90, 85, 80, 70, 50, 80, 50, 80, 85, 90, 90, 95, 80, 85, 90, 90, 85, 80, 80, 80, 80, 75, 80, 50, 80, 60, 95, 80, 95],
    'W': [95, 85, 90, 75, 70, 80, 65, 90, 90, 50, 70, 80, 90, 75, 75, 80, 95, 90, 60, 95, 80, 50, 80, 70, 80, 80, 50, 85, 90, 90, 85, 80, 70, 80],
    'NA': [89.5, 88, 80.5, 79.5, 71.5, 80, 63.5, 92, 76.5, 54.5, 76.5, 67, 73, 82.5, 77.5, 80.5, 76, 73, 78, 74.5, 88.5, 68, 74, 73.5, 72.5, 81, 63, 86.5, 72, 89.5, 74, 74.5, 74, 80]}
df = pd.DataFrame(data)
X = df[['KD', 'BI', 'K', 'W']].values
y = df['NA'].values

kf = KFold(n_splits=10, shuffle=True, random_state=0)
weights = []
biases = []
mse_scores = []

for train_index, test_index in kf.split(X):
    X_train, X_test = X[train_index], X[test_index]
    y_train, y_test = y[train_index], y[test_index]
    scaler_X = StandardScaler()
    scaler_y = StandardScaler()
    X_train_scaled = scaler_X.fit_transform(X_train)
    X_test_scaled = scaler_X.transform(X_test)
    y_train_scaled = scaler_y.fit_transform(y_train.reshape(-1, 1)).flatten()
    model = Sequential()
    model.add(Dense(1, input_dim=4, activation='linear'))
    opt = Adam(learning_rate=0.001)
    model.compile(optimizer=opt, loss='mse')
    es = EarlyStopping(monitor='loss', patience=20, restore_best_weights=True)
    model.fit(X_train_scaled, y_train_scaled, epochs=1000, verbose=0, callbacks=[es])
    y_pred_scaled = model.predict(X_test_scaled).flatten()
    y_pred = scaler_y.inverse_transform(y_pred_scaled.reshape(-1, 1)).flatten()
    mse = mean_squared_error(y_test, y_pred)
    mse_scores.append(mse)
    weights.append(model.get_weights()[0].flatten())
    biases.append(model.get_weights()[1][0])

b1, b2, b3, b4 = np.mean(weights, axis=0)
bias = np.mean(biases)
print(f'Rata-rata bobot (b1, b2, b3, b4): {b1:.4f}, {b2:.4f}, {b3:.4f}, {b4:.4f}')
print(f'Rata-rata bias: {bias:.4f}')
print('Rata-rata MSE:', np.mean(mse_scores))

scaler_X = StandardScaler()
scaler_y = StandardScaler()
X_scaled = scaler_X.fit_transform(X)
y_scaled = scaler_y.fit_transform(y.reshape(-1, 1)).flatten()
model = Sequential()
model.add(Dense(1, input_dim=4, activation='linear'))
opt = Adam(learning_rate=0.001)
model.compile(optimizer=opt, loss='mse')
es = EarlyStopping(monitor='loss', patience=20, restore_best_weights=True)
model.fit(X_scaled, y_scaled, epochs=1000, verbose=0, callbacks=[es])
w = model.get_weights()[0].flatten()
b = model.get_weights()[1][0]

na_predict_scaled = model.predict(X_scaled).flatten()
na_predict = scaler_y.inverse_transform(na_predict_scaled.reshape(-1, 1)).flatten()

print(f'Rumus Nilai Akhir (NA) dalam skala normalisasi: NA = {w[0]:.4f}*KD + {w[1]:.4f}*BI + {w[2]:.4f}*K + {w[3]:.4f}*W + {b:.4f}')
tabel = pd.DataFrame({
    'NO': np.arange(1, len(df)+1),
    'KD': df['KD'],
    'BI': df['BI'],
    'K': df['K'],
    'W': df['W'],
    'NA (original)': df['NA'],
    'NA (predict)': na_predict.round(2)
})
print(tabel.to_string(index=False)) 

1/1 [==============================] - 0s 64ms/step
Rata-rata bobot (b1, b2, b3, b4): 0.3997, 0.3143, 0.2011, 0.2366
Rata-rata bias: -0.0000
Rata-rata MSE: 32.00765555684278
2/2 [==============================] - 0s 5ms/step
Rumus Nilai Akhir (NA) dalam skala normalisasi: NA = 0.7428*KD + 0.1674*BI + -0.0788*K + 0.3649*W + 0.0012
 NO  KD  BI  K  W  NA (original)  NA (predict)
  1  90  80 90 95           89.5     85.529999
  2  85  90 95 85           88.0     81.830002
  3  90  75 80 90           80.5     84.410004
  4  80  80 85 75           79.5     77.099998
  5  75  50 90 70           71.5     70.940002
  6  90  80 90 80           80.0     81.959999
  7  50  60 85 65           63.5     61.070000
  8 100  95 80 90           92.0     90.180000
  9  85  80 70 90           76.5     83.419998
 10  50  55 50 50           54.5     58.889999
 11  75  85 80 70           76.5     74.660004
 12  80  45 50 80           67.0     76.940002
 13  50  75 80 90           73.0     68.650002
 14  90  8